In [82]:
MODEL="qwen2.5-1.5b"
TASK="countdown"
N=1

In [ ]:
SOLVER_FILE = f"artifacts/{TASK}/models/method_ac_models/{MODEL}/evals/eval__all-hint-levels.internal.json"
VERIFIER_FILE = f"artifacts/{TASK}/models/method_a_predictors/{MODEL}/evals/eval__verifier-decisions.internal.json"


In [84]:
import json
from pathlib import Path


def load_json(path):
    with open(path) as f:
        return json.load(f)

solver_results = load_json(SOLVER_FILE)
verifier_results = load_json(VERIFIER_FILE)

solver_details = solver_results["details"]
verifier_details = verifier_results["details"]

print(f"Solver:   {len(solver_details)} details, keys={list(solver_details[0].keys())}")
print(f"Verifier: {len(verifier_details)} details, keys={list(verifier_details[0].keys())}")
print(f"Solver multisample:   {'samples' in solver_details[0]}")
print(f"Verifier multisample: {'samples' in verifier_details[0]}")


Solver:   7499 details, keys=['index', 'variant', 'level', 'ground_truth', 'n_samples', 'n_correct', 'samples', 'source_index', 'hint_level', 'hint_sequence', 'split']
Verifier: 7499 details, keys=['index', 'variant', 'level', 'ground_truth', 'n_samples', 'n_correct', 'samples', 'source_index', 'hint_level', 'hint_sequence', 'split']
Solver multisample:   True
Verifier multisample: True


In [85]:
def group_by_index(json_big):
    new_json_big = {}
    for i, datapoint in enumerate(json_big):
        source_index = datapoint["source_index"]
        if source_index not in new_json_big.keys():
            new_json_big[source_index] = {}

        assert datapoint["hint_level"] not in new_json_big[source_index].keys()
        new_json_big[source_index][datapoint["hint_level"]] = datapoint

    return new_json_big
        

In [86]:
solver_details_by_index = group_by_index(solver_details)
verifier_details_by_index = group_by_index(verifier_details)

In [87]:
assert solver_details_by_index.keys() == verifier_details_by_index.keys(), (
    f"Mismatch: solver-only={set(solver_details_by_index) - set(verifier_details_by_index)}, "
    f"verifier-only={set(verifier_details_by_index) - set(solver_details_by_index)}"
)

In [88]:
solver_details_by_index[3]

{0: {'index': 18,
  'variant': '5_operands',
  'level': 'unknown',
  'ground_truth': {'target': 41,
   'numbers': [70, 18, 80, 59, 26],
   'answer': '((70 + 18) - (80 - (59 - 26)))',
   'hint_exprs': ['(70 + 18)',
    '(59 - 26)',
    '(80 - (59 - 26))',
    '((70 + 18) - (80 - (59 - 26)))'],
   'variant': '5_operands',
   'hints': ['(70 + 18)',
    '(59 - 26)',
    '(80 - (59 - 26))',
    '((70 + 18) - (80 - (59 - 26)))']},
  'n_samples': 32,
  'n_correct': 30,
  'samples': [{'generation': " First, I need to use the numbers [70, 18, 80, 59, 26] exactly once each with basic operations to get 41.\n\nHmm, 70 is a large number. If I can somehow use it in a way that results in a number close to 41... Maybe I can do something like 70 - (something) = 41? That would mean I need to get 29 from the other numbers. Let me see: 80 - 59 = 21, 26 - 18 = 8, 21 + 8 = 29. Wait, that gets me 29, and 70 - 29 = 41! Let me check that again.\n\nSo, if I do (70 - ((80 - 59) + (26 - 18)))... Let me calculate 

In [89]:
import random

data_final = []

for source_index in solver_details_by_index.keys():
    solver_data = solver_details_by_index[source_index]
    verifier_data = verifier_details_by_index[source_index]

    assert len(verifier_data) == len(solver_data)

    hint_levels_max = len(verifier_data)
    if TASK == "countdown":
        difficulty_level = solver_data[0]['variant']
    elif TASK == "competition_math":
        difficulty_level = solver_data[0]["level"].split(' ')[1]

    for _ in range(N):

        hint_level = 0
        while hint_level < hint_levels_max:
            rand_n = random.randrange(len(verifier_data[hint_level]['samples']))
            verifier_sample = verifier_data[hint_level]['samples'][rand_n]['predicted_answer']
            if verifier_sample == '1':
                break

            hint_level += 1

        hint_level = min(hint_levels_max -1, hint_level)

        rand_n = random.randrange(len(solver_data[hint_level]['samples']))
        solver_solution = solver_data[hint_level]['samples'][rand_n]['correct']

        data_final.append({
            "variant": difficulty_level,
            "num_hints": hint_level,
            "correct": solver_solution
        }
        )

In [90]:
import pandas as pd

df = pd.DataFrame(data_final)

# Raw counts per variant x hint level
counts_table = pd.crosstab(df["variant"], df["num_hints"])
counts_table["total"] = counts_table.sum(axis=1)
counts_table = counts_table[["total"] + [c for c in counts_table.columns if c != "total"]]

# Correct counts per variant x hint level (correct is bool -> sum counts True)
correct_table = pd.crosstab(df["variant"], df["num_hints"], values=df["correct"], aggfunc="sum").fillna(0).astype(int)
correct_table["total"] = correct_table.sum(axis=1)
correct_table = correct_table[["total"] + [c for c in correct_table.columns if c != "total"]]

# Accuracy = correct / count, cell-by-cell
accuracy_table = (correct_table / counts_table).round(4)

print("Counts:")
display(counts_table)
print("\nCorrect:")
display(correct_table)
print("\nAccuracy:")
display(accuracy_table)

Counts:


num_hints,total,0,1,2,3,4,5
variant,,,,,,,
4_operands,507,487,20,0,0,0,0
5_operands,487,441,38,8,0,0,0
6_operands,506,259,123,73,33,17,1



Correct:


num_hints,total,0,1,2,3,4,5
variant,,,,,,,
4_operands,460,442,18,0,0,0,0
5_operands,326,289,30,7,0,0,0
6_operands,243,105,58,46,18,15,1



Accuracy:


num_hints,total,0,1,2,3,4,5
variant,,,,,,,
4_operands,0.9073,0.9076,0.9000,NaN,NaN,NaN,NaN
5_operands,0.6694,0.6553,0.7895,0.8750,NaN,NaN,NaN
6_operands,0.4802,0.4054,0.4715,0.6301,0.5455,0.8824,1.0


In [91]:
# Write actual .tsv files to disk instead of relying on copy/paste from
# the notebook's rendered output (browsers can mangle tabs/whitespace when
# copying from an output cell). Open these files and copy-paste from a
# plain text editor, or use Google Sheets/Excel's File > Import instead.
counts_table.to_csv("counts_table.tsv", sep="\t")
correct_table.to_csv("correct_table.tsv", sep="\t")
accuracy_table.to_csv("accuracy_table.tsv", sep="\t")
print("Wrote counts_table.tsv, correct_table.tsv, accuracy_table.tsv")

Wrote counts_table.tsv, correct_table.tsv, accuracy_table.tsv
